In [ ]:
# =========================================================
# Museum Guardrail 一次貼上可跑完版（含軟限制開關、抗429、Grid Search、3類TPR/TNR）
# =========================================================

# 可選：!pip -q install --upgrade openai pandas scikit-learn pydantic tqdm python-dotenv

import os, json, time, random, asyncio, statistics
from collections import Counter
from typing import Literal, List, Dict
import pandas as pd
from pydantic import BaseModel

# ---------- OpenAI Client ----------
from openai import OpenAI
from openai import RateLimitError, APIError, APIConnectionError, BadRequestError

# ============ 你只需要改這裡（若用你自己的檔） ============
FILE_ID   = "1ZJRH_V97vvmj4QTJW3rI4zAwLgdzEW2N"   # 你的 Google Drive File ID
FILENAME  = "guardrail_questions_labeled (1).csv" # 檔名（下載後會存這個名字）
# ========================================================

# ---------- API Key 設定（Colab Variables 優先） ----------
api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key:
    try:
        from google.colab import userdata  # type: ignore
        api_key = userdata.get("OPENAI_API_KEY") or ""
    except Exception:
        pass
assert api_key, "請先設定 OPENAI_API_KEY（Colab Variables 或環境變數）。"
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()

# ---------- 全域參數（抗429 / few-shots / 模型） ----------
MODEL_DEFAULT        = "gpt-4.1-mini"   # 推論模型
MAX_FEWSHOTS         = 12               # few-shots 上限（省 token）
MAX_TOKENS_OUT       = 60               # 只需輸出 JSON
SLEEP_BETWEEN_VOTES  = 0.35             # 投票的每一票間隔
SLEEP_BETWEEN_ITEMS  = 0.12             # 題與題間隔
MAX_RETRIES          = 6                # 自動重試次數
BACKOFF_BASE         = 0.8              # 回退起點
BACKOFF_CAP          = 20.0             # 回退上限
DEV_SAMPLE_K         = None             # 例如 30；None 表示不抽樣

# ---------- 軟限制/提示（不想限制就關掉） ----------
USE_RULE_HINTS = "off"  # "off" | "soft"
EXHIBIT_CUES = {"壺","杯","碗","碟","瓶","釉","瓷","玉","鼻煙壺","蓋碗","指甲套","懷錶","佩","佛手"}
LIFESTYLE_TERMS = {"路線","票價","售票","餐廳","咖啡","天氣","手機","充電","廁所","營業時間"}

def hint_vote_for(query:str):
    """soft 模式：生活用語且沒有展品線索 → 幫 N 加一張『軟票』；其他不干預"""
    if USE_RULE_HINTS != "soft":
        return None
    has_exhibit = any(t in query for t in EXHIBIT_CUES)
    has_life    = any(t in query for t in LIFESTYLE_TERMS)
    if has_life and not has_exhibit:
        return "N"
    return None

# =========================================================
# 讀檔 & 整理資料
# =========================================================
def _try_read_csv(paths, **kwargs):
    for p in paths:
        if os.path.exists(p):
            print(f"✅ 讀到本機檔：{p}")
            return pd.read_csv(p, **kwargs)
    return None

def load_dataset_from_drive(file_id:str, filename:str) -> pd.DataFrame:
    # A) 嘗試本機路徑
    df = _try_read_csv([f"/content/{filename}", filename])
    # B) 嘗試 MyDrive 常見路徑
    if df is None:
        md_paths = [
            f"/content/drive/MyDrive/{filename}",
            f"/content/drive/MyDrive/datasets/{filename}",
        ]
        df = _try_read_csv(md_paths)
    # C) 都沒有 → gdown 下載
    if df is None:
        print("⚠️ 本機/MyDrive 未找到檔案，改用 gdown 下載…")
        try:
            import gdown
        except Exception:
            import sys, subprocess
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
            import gdown
        url = f"https://drive.google.com/uc?id={file_id}"
        out = f"/content/{filename}"
        gdown.download(url, out, quiet=False)
        df = pd.read_csv(out)
    return df

def coerce_label_to_bool(s: pd.Series) -> pd.Series:
    """把 'True'/'False'/1/0/yes/no → bool"""
    if s.dtype == "bool":
        return s
    strv = s.astype(str).str.strip().str.lower()
    mapping = {
        "true": True, "1": True, "yes": True, "y": True, "t": True,
        "false": False,"0": False,"no": False,"n": False,"f": False
    }
    return strv.map(mapping).fillna(False).astype(bool)

def prepare_splits(df: pd.DataFrame):
    # 標準欄位：query + suppose_answer（Y/F/N）
    if "query" not in df.columns:
        for alt in ["text","question","prompt"]:
            if alt in df.columns:
                df = df.rename(columns={alt:"query"})
                break
    assert "query" in df.columns, "找不到 'query' 欄位。"

    # suppose_answer 沒有時，嘗試用 query_type 補
    if "suppose_answer" not in df.columns:
        if "query_type" in df.columns:
            df["suppose_answer"] = df["query_type"].astype(str).str.upper()
        else:
            raise ValueError("找不到 'suppose_answer' 或 'query_type' 欄位。")

    # 清理 Y/F/N
    df["suppose_answer"] = df["suppose_answer"].astype(str).str.strip().str.upper()
    df = df[df["suppose_answer"].isin(["Y","F","N"])].copy()

    # 額外欄位（可有可無）
    if "label" in df.columns:
        df["label"] = coerce_label_to_bool(df["label"])
    if "query_type" not in df.columns:
        df["query_type"] = df["suppose_answer"]

    print(f"資料集總共有 {len(df)} 筆資料")
    print("欄位：", list(df.columns))
    print("類別分布（suppose_answer）：")
    print(df["suppose_answer"].value_counts())

    # 分層抽樣：train 15% / dev 50% / test 35%
    from sklearn.model_selection import train_test_split
    total = len(df)
    train_df, temp_df = train_test_split(
        df, test_size=0.85, stratify=df["suppose_answer"], random_state=42
    )
    dev_df, test_df = train_test_split(
        temp_df, test_size=35/85, stratify=temp_df["suppose_answer"], random_state=42
    )

    # 轉 list[dict] 給推論
    return (
        train_df.to_dict("records"),
        dev_df.to_dict("records"),
        test_df.to_dict("records"),
        train_df, dev_df, test_df
    )

# =========================================================
# few-shots & Prompt 生成
# =========================================================
def build_fewshots(dataset, k=MAX_FEWSHOTS):
    rows = dataset[:k] if len(dataset) > k else dataset
    return "\n".join([f"Query: {x['query']}\nOutput: {x['suppose_answer']}" for x in rows])

def make_prompts(FEWSHOTS: str):
    BASE_CONTEXT = """
你是《紅樓夢》主題展的導覽問答分類器，負責把觀眾提問分為三類：
- Y（事實）：可由展品標示／材質／用途／年代／製作技法／稱謂，或《紅樓夢》文本中的具體資訊直接回答。
- F（詮釋）：需要結合象徵、情感、社會文化脈絡做合理解讀或想像（一般觀眾可理解，不是艱深學術）。
- N（無關）：與《紅樓夢》或清單內展品無關；或只是動線、天氣、餐飲、手機等生活問題。
""".strip()

    PROMPT_BASE = f"""
{BASE_CONTEXT}

# 輸出格式（只允許 JSON）
{{
  "category": "Y" | "F" | "N",
  "rejection_answer": "若為N，請用台灣繁體中文友善簡短婉拒；若非N，請輸出空字串"
}}

# 判斷要點
- 問句若同時有「用途/材質（像事實）」但核心在「感受/象徵/像誰」→ 判 F。
- 純展場動線/販售/天氣/拍照規定等 → 判 N。
- 僅出現「紅樓夢」字樣但實際問生活瑣事 → 判 N。
- Y/F/N 只能擇一，禁止輸出其他文字。

# Few-shot（學習格式，不要複製到輸出）
{FEWSHOTS}
""".strip()

    PROMPT_STRICT = f"""
{BASE_CONTEXT}

# 嚴格判定規則
1) 句子含「象徵/感覺/比較像/氛圍/像某角色」→ 優先判 F，除非 100% 明確在問尺寸/材質/年代等可驗證事實。
2) 問題以「這附近/今天/可以/手機/拍照/販售/路線/票價」為核心 → 判 N。
3) 用「是不是」「會不會」但其實在問用途、材質、製程且可查證 → 判 Y。
4) 分不清時，先檢查是否可由展品標示直接回答：可以 → Y；不行但仍與展品詮釋有關 → F；否則 → N。

# 僅輸出以下 JSON：
{{"category":"Y|F|N","rejection_answer":"N時必填，其餘留空"}}

# Few-shot（學習格式，不要複製到輸出）
{FEWSHOTS}
""".strip()

    PROMPT_BOUNDARY = f"""
{BASE_CONTEXT}

# 決策表（依序檢查）
- 若問題核心為「路線/天氣/販售/票/拍照/手機/餐飲」→ N
- 否則若可由展籤、材質、用途、年代、技法、稱謂直接作答 → Y
- 否則若與展品象徵或《紅樓夢》角色／情境的詮釋連結明顯 → F
- 其餘 → N

# 常見誤判校正
- 「這個紋樣是不是比較內斂？」→ F（詮釋）
- 「這個杯子通常搭什麼托？」→ Y（事實）
- 「這裡有賣《紅樓夢》周邊嗎？」→ N（無關）

# 僅輸出以下 JSON：
{{"category":"Y"|"F"|"N","rejection_answer":"N時提供友善婉拒；Y/F 時為空字串"}}

# Few-shot（學習格式，不要複製到輸出）
{FEWSHOTS}
""".strip()

    return PROMPT_BASE, PROMPT_STRICT, PROMPT_BOUNDARY

# =========================================================
# OpenAI 呼叫（含回退重試）
# =========================================================
async def _openai_chat_with_retry(messages, model=MODEL_DEFAULT,
                                  temperature=0.3, max_tokens=MAX_TOKENS_OUT):
    backoff = BACKOFF_BASE
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = await asyncio.to_thread(
                client.chat.completions.create,
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp
        except (RateLimitError, APIError, APIConnectionError):
            if attempt == MAX_RETRIES:
                raise
            jitter = random.uniform(0.1, 0.5)
            await asyncio.sleep(min(BACKOFF_CAP, backoff) + jitter)
            backoff *= 1.6
        except BadRequestError:
            raise

class MuseumGuardrailResult(BaseModel):
    category: Literal['Y','F','N']
    rejection_answer: str = ""

async def _call_once(query: str, instructions: str,
                     model: str = MODEL_DEFAULT,
                     temperature: float = 0.3) -> MuseumGuardrailResult:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": (
            '請只輸出 JSON，格式：{"category":"Y|F|N","rejection_answer":"N時填，Y/F留空"}\n\n'
            "題目：\n" + query
        )}
    ]
    resp = await _openai_chat_with_retry(messages, model=model,
                                         temperature=temperature,
                                         max_tokens=MAX_TOKENS_OUT)
    raw = resp.choices[0].message.content.strip()

    # 解析 JSON（保險切片）
    try:
        data = json.loads(raw)
    except Exception:
        s = raw[raw.find("{"): raw.rfind("}")+1]
        data = json.loads(s) if ("{" in raw and "}" in raw) else {}

    cat = str(data.get("category", "N")).upper()
    if cat not in ("Y","F","N"):
        cat = "N"
    rej = data.get("rejection_answer") or ""
    if cat == "N" and not rej:
        rej = "這題和展品／《紅樓夢》沒有直接關聯，暫時無法提供解說喔～"
    return MuseumGuardrailResult(category=cat, rejection_answer=rej)

async def predict_with_agent(query: str, instructions: str,
                             model: str = MODEL_DEFAULT,
                             n_votes: int = 1, temperature: float = 0.3,
                             tie_break_order = ("N","Y","F")) -> MuseumGuardrailResult:
    n_votes = max(1, int(n_votes))
    votes = []
    for _ in range(n_votes):
        out = await _call_once(query, instructions, model=model, temperature=temperature)
        votes.append(out.category)
        if n_votes > 1:
            await asyncio.sleep(SLEEP_BETWEEN_VOTES)

    # 軟提示投一票（可關）
    hint = hint_vote_for(query)
    if hint:
        votes.append(hint)

    # 多數決 + 同票偏好
    rank = Counter(votes).most_common()
    if len(rank) == 1 or rank[0][1] != rank[1][1]:
        final_cat = rank[0][0]
    else:
        for cand in tie_break_order:
            if any(k == cand and v == rank[0][1] for k, v in rank):
                final_cat = cand
                break

    rej = "" if final_cat in ("Y","F") else "這題和展品／《紅樓夢》沒有直接關聯，暫時無法提供解說喔～"
    return MuseumGuardrailResult(category=final_cat, rejection_answer=rej)

# =========================================================
# 指標計算（Accuracy / Macro TPR / Macro TNR / 混淆矩陣）
# =========================================================
CLASSES = ["Y","F","N"]

def _compute_metrics(y_true: List[str], y_pred: List[str], title="Dataset", verbose=True):
    df = pd.DataFrame({"true": y_true, "pred": y_pred})
    cm = pd.crosstab(df["true"], df["pred"], dropna=False).reindex(index=CLASSES, columns=CLASSES, fill_value=0)
    metrics = []
    total = len(df)
    for c in CLASSES:
        TP = cm.loc[c, c]
        FP = cm[c].sum() - TP
        FN = cm.loc[c].sum() - TP
        TN = total - TP - FP - FN
        tpr = (TP / (TP + FN)) if (TP + FN) else 0.0
        tnr = (TN / (TN + FP)) if (TN + FP) else 0.0
        prec = (TP / (TP + FP)) if (TP + FP) else 0.0
        f1 = (2*prec*tpr)/(prec+tpr) if (prec+tpr) else 0.0
        metrics.append({"class": c, "TP":TP, "FP":FP, "TN":TN, "FN":FN,
                        "TPR(recall)%": round(tpr*100,2), "TNR(specificity)%": round(tnr*100,2),
                        "Precision%": round(prec*100,2), "F1%": round(f1*100,2),
                        "Support": int(cm.loc[c].sum())})
    acc = (df["true"] == df["pred"]).mean()*100
    macro_tpr = statistics.mean([m["TPR(recall)%"] for m in metrics])
    macro_tnr = statistics.mean([m["TNR(specificity)%"] for m in metrics])

    if verbose:
        print(f"\n=== {title}（{total} 筆）===")
        print(f"Accuracy: {acc:.2f}%")
        print(f"Macro TPR(Recall): {macro_tpr:.2f}%")
        print(f"Macro TNR(Specificity): {macro_tnr:.2f}%\n")
        print("Per-class(metrics):")
        print(pd.DataFrame(metrics))
        print("\nConfusion Matrix (rows=true, cols=pred):")
        print(cm.rename(columns={c:f"pred_{c}" for c in CLASSES}))
    return {"acc": acc, "macro_tpr": macro_tpr, "macro_tnr": macro_tnr, "metrics": metrics, "cm": cm,
            "y_true": y_true, "y_pred": y_pred}

async def evaluate_dataset_async(dataset, predict_fn, title="Dataset", verbose=True):
    if DEV_SAMPLE_K and len(dataset) > DEV_SAMPLE_K:
        dataset = dataset[:DEV_SAMPLE_K]
    y_true, y_pred = [], []
    for i, item in enumerate(dataset, 1):
        out = await predict_fn(item["query"])
        y_true.append(item["suppose_answer"])
        y_pred.append(out.category)
        await asyncio.sleep(SLEEP_BETWEEN_ITEMS)
    return _compute_metrics(y_true, y_pred, title=title, verbose=verbose)

# =========================================================
# Grid Search（Prompt × 溫度 × 投票 × 同票偏好）
# =========================================================
async def grid_search_on_dev(dev_dataset, prompts):
    PROMPT_BASE, PROMPT_STRICT, PROMPT_BOUNDARY = prompts
    VARIANTS = [
        {"name":"strict_t0.3_v3_biasN",   "prompt":PROMPT_STRICT,   "model":MODEL_DEFAULT, "temperature":0.3, "n_votes":3, "tie_break":("N","Y","F")},
        {"name":"boundary_t0.4_v3_biasN", "prompt":PROMPT_BOUNDARY, "model":MODEL_DEFAULT, "temperature":0.4, "n_votes":3, "tie_break":("N","Y","F")},
        {"name":"boundary_t0.4_v5_biasN", "prompt":PROMPT_BOUNDARY, "model":MODEL_DEFAULT, "temperature":0.4, "n_votes":5, "tie_break":("N","Y","F")},
        {"name":"base_t0.5_v5_biasY",     "prompt":PROMPT_BASE,     "model":MODEL_DEFAULT, "temperature":0.5, "n_votes":5, "tie_break":("Y","F","N")},
    ]
    best = None
    results_summary = []
    for cfg in VARIANTS:
        print(f"\n>>> 試跑 {cfg['name']} (model={cfg['model']}, temp={cfg['temperature']}, votes={cfg['n_votes']}, tie={cfg['tie_break']})")
        async def _predict(q):
            return await predict_with_agent(
                q, instructions=cfg["prompt"], model=cfg["model"],
                n_votes=cfg["n_votes"], temperature=cfg["temperature"],
                tie_break_order=cfg["tie_break"]
            )
        try:
            res = await evaluate_dataset_async(dev_dataset, _predict, title=cfg["name"], verbose=True)
        except RateLimitError:
            print("⚠️ 觸發 429，已跳過該組合（可降低 n_votes/FEWSHOTS 或提高 SLEEP_*）")
            continue
        score = min(res["macro_tpr"], res["macro_tnr"])
        results_summary.append((cfg, res, score))
        if (best is None) or (score > best[2]):
            best = (cfg, res, score)
        await asyncio.sleep(1.0)

    if not best:
        raise RuntimeError("所有組合都因 429 失敗；請下修 FEWSHOTS / n_votes 或開啟 DEV_SAMPLE_K。")

    print("\n=== Dev Grid Search 摘要（以 min(TPR,TNR) 排序）===")
    for cfg, res, score in sorted(results_summary, key=lambda x: x[2], reverse=True):
        print(f"{cfg['name']}: Acc={res['acc']:.2f}  MacroTPR={res['macro_tpr']:.2f}  MacroTNR={res['macro_tnr']:.2f}  score={score:.2f}")
    print(f"\n✅ 最佳組合：{best[0]['name']}（min(TPR,TNR)={best[2]:.2f}）")
    return best  # (cfg, res, score)

# =========================================================
# 主流程（一次跑完）
# =========================================================
async def main():
    # 1) 讀檔
    df = load_dataset_from_drive(FILE_ID, FILENAME)

    # 2) 準備 splits
    train_dataset, dev_dataset, test_dataset, train_df, dev_df, test_df = prepare_splits(df)

    # 3) few-shots & prompts
    FEWSHOTS = build_fewshots(train_dataset, k=MAX_FEWSHOTS)
    prompts = make_prompts(FEWSHOTS)

    # 4) Dev 上挑最佳
    #    想先省額度，可先開：DEV_SAMPLE_K = 30（跑完OK再設回 None）
    best_cfg, dev_res, best_score = await grid_search_on_dev(dev_dataset, prompts)

    # 5) 用最佳組合跑 Test
    print("\n\n=== 用最佳 Prompt 評估 Test ===")
    async def _predict_best(q):
        return await predict_with_agent(
            q, instructions=best_cfg["prompt"], model=best_cfg["model"],
            n_votes=best_cfg["n_votes"], temperature=best_cfg["temperature"],
            tie_break_order=best_cfg["tie_break"]
        )
    _ = await evaluate_dataset_async(test_dataset, _predict_best, title=f"TEST with {best_cfg['name']}", verbose=True)

# Colab 不能用 asyncio.run 時，請把下一行改成：await main()
import sys
if "google.colab" in sys.modules:
    # Colab 支援 top-level await
    await main()
else:
    # 一般 Python 腳本環境
    asyncio.run(main())


⚠️ 本機/MyDrive 未找到檔案，改用 gdown 下載…


Downloading...
From: https://drive.google.com/uc?id=1ZJRH_V97vvmj4QTJW3rI4zAwLgdzEW2N
To: /content/guardrail_questions_labeled (1).csv
100%|██████████| 15.2k/15.2k [00:00<00:00, 29.9MB/s]


資料集總共有 142 筆資料
欄位： ['query_type', 'query', 'suppose_answer', 'predict_answer', 'rejection_content', 'label']
類別分布（suppose_answer）：
suppose_answer
Y    51
F    50
N    41
Name: count, dtype: int64

>>> 試跑 strict_t0.3_v3_biasN (model=gpt-4.1-mini, temp=0.3, votes=3, tie=('N', 'Y', 'F'))

=== strict_t0.3_v3_biasN（71 筆）===
Accuracy: 77.46%
Macro TPR(Recall): 78.67%
Macro TNR(Specificity): 88.41%

Per-class(metrics):
  class  TP  FP  TN  FN  TPR(recall)%  TNR(specificity)%  Precision%     F1%  \
0     Y  14   5  41  11          56.0              89.13       73.68   63.64   
1     F  20  11  35   5          80.0              76.09       64.52   71.43   
2     N  21   0  50   0         100.0             100.00      100.00  100.00   

   Support  
0       25  
1       25  
2       21  

Confusion Matrix (rows=true, cols=pred):
pred  pred_Y  pred_F  pred_N
true                        
Y         14      11       0
F          5      20       0
N          0       0      21

>>> 試跑 boundary_t0.4_v3